In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('loan_data.csv')
df.head()

,applicant_id,name,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,APP1170,Raj Smith,NaN,"$66,400",NaN,89100.0,Unemployed,No,Female,Rejected
1,APP1177,Omar Silva,25.0,"$75,700",645.0,228600.0,Employed,NaN,Male,Approved
2,APP1132,Mei Khan,24.0,"$26,400",580.0,5000.0,employed,Yes,Female,Approved
3,APP1193,Liam Ali,28.0,"$78,500",NaN,272600.0,Employed,No,Female,Approved
4,APP1102,Raj Kumar,35.0,NaN,573.0,159000.0,employed,N,Female,Rejected


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 212 entries, 0 to 211
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   applicant_id       212 non-null    str    
 1   name               212 non-null    str    
 2   age                194 non-null    float64
 3   income             202 non-null    str    
 4   credit_score       195 non-null    float64
 5   loan_amount        212 non-null    float64
 6   employment_status  201 non-null    str    
 7   married            200 non-null    str    
 8   gender             202 non-null    str    
 9   approved           212 non-null    str    
dtypes: float64(3), str(7)
memory usage: 26.2 KB


In [4]:
df.isna().sum()

applicant_id          0
name                  0
age                  18
income               10
credit_score         17
loan_amount           0
employment_status    11
married              12
gender               10
approved              0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(12)

In [6]:
df = df.drop(columns=['applicant_id','name'])
df.columns

Index(['age', 'income', 'credit_score', 'loan_amount', 'employment_status',
       'married', 'gender', 'approved'],
      dtype='str')

In [7]:
len(df)
df = df.drop_duplicates()
len(df)


200

In [8]:
print(df.income.head().tolist())

df['income'] = (
   df['income'].astype(str)
    .str.replace(r'[$,]', '', regex=True)
    .str.strip()
)
df['income'] = pd.to_numeric(df['income'], errors='coerce')

print(df.income.head().tolist())
print(df.income.dtype)

['$66,400', '$75,700', '$26,400', '$78,500', nan]
[66400.0, 75700.0, 26400.0, 78500.0, nan]
float64


In [9]:
df.gender.value_counts(dropna=False)


gender
Male      37
M         34
female    34
male      31
F         28
Female    26
NaN       10
Name: count, dtype: int64

In [10]:
df.gender = df.gender.astype(str).str.strip().str.lower().map(
    {'male': 'Male', 'm': 'Male', 'female':'Female', 'f':'Female'}
)

df.married = df.married.astype(str).str.strip().str.lower().map(
     {'yes': 'Yes', 'y': 'Yes', 'no': 'No', 'n': 'No'}
)

emp_map = {
    'employed': 'Employed',
    'self employed': 'Self-Employed', 'self-employed': 'Self-Employed',
    'unemployed': 'Unemployed'
}

df.employment_status = df.employment_status.astype(str).str.strip().str.lower().map(emp_map)

In [11]:
df.gender.value_counts(dropna=False)

gender
Male      102
Female     88
NaN        10
Name: count, dtype: int64

In [12]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,184.0,43.527174,21.366674,-5.0,31.0,41.0,54.00,200.0
income,188.0,52789.361702,21143.955119,12000.0,37925.0,51150.0,68325.00,104000.0
credit_score,184.0,645.951087,134.420646,0.0,581.5,656.0,729.25,850.0
loan_amount,200.0,151629.500000,86473.742375,5000.0,88250.0,157450.0,205700.00,448800.0


In [13]:
df.age.count(), df.credit_score.count()

(np.int64(184), np.int64(184))

In [14]:
df.loc[(df.age < 18) | (df.age > 100), 'age'] = np.nan
df.loc[(df.credit_score < 300) | (df.credit_score > 850), 'credit_score'] = np.nan

In [15]:
df.age.count(),df.credit_score.count()

(np.int64(180), np.int64(180))

In [16]:
print(df.isna().sum())

age                  20
income               12
credit_score         20
loan_amount           0
employment_status    10
married              10
gender               10
approved              0
dtype: int64


In [17]:
num_cols = ['age','income','credit_score','loan_amount']
categorical_cols = ['employment_status','married','gender']

for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

for c in categorical_cols:
    df[c] = df[c].fillna(df[c].mode()[0])


In [18]:
print(df.isna().sum())

age                  0
income               0
credit_score         0
loan_amount          0
employment_status    0
married              0
gender               0
approved             0
dtype: int64


In [19]:
df.approved.value_counts()

approved
Rejected    110
Approved     90
Name: count, dtype: int64

In [20]:
df.head()

,age,income,credit_score,loan_amount,employment_status,married,gender,approved
0,41.0,66400.0,665.5,89100.0,Unemployed,No,Female,Rejected
1,25.0,75700.0,645.0,228600.0,Employed,Yes,Male,Approved
2,24.0,26400.0,580.0,5000.0,Employed,Yes,Female,Approved
3,28.0,78500.0,665.5,272600.0,Employed,No,Female,Approved
4,35.0,51150.0,573.0,159000.0,Employed,No,Female,Rejected


In [21]:
df['approved'] = df['approved'].map({'Approved': 1, 'Rejected': 0})

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

df.head()

,age,income,credit_score,loan_amount,approved,employment_status_Self-Employed,employment_status_Unemployed,married_Yes,gender_Male
0,41.0,66400.0,665.5,89100.0,0,False,True,False,False
1,25.0,75700.0,645.0,228600.0,1,False,False,True,True
2,24.0,26400.0,580.0,5000.0,1,False,False,True,False
3,28.0,78500.0,665.5,272600.0,1,False,False,False,False
4,35.0,51150.0,573.0,159000.0,0,False,False,False,False


In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

df[num_cols].head()

,age,income,credit_score,loan_amount
0,-0.098160,0.670402,0.052355,-0.724918
1,-1.412433,1.125193,-0.176488,0.892336
2,-1.494575,-1.285690,-0.902087,-1.699909
3,-1.166007,1.262119,0.052355,1.402438
4,-0.591012,-0.075358,-0.980228,0.085448


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = df.drop('approved', axis='columns')
y = df.approved

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

print(f'Test Accuracy: {model.score(X_test, y_test)}')

Test Accuracy: 0.75


In [35]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(f"Classification Report\n\n {classification_report(y_test, y_pred)}")

Classification Report

               precision    recall  f1-score   support

           0       0.78      0.78      0.78        23
           1       0.71      0.71      0.71        17

    accuracy                           0.75        40
   macro avg       0.74      0.74      0.74        40
weighted avg       0.75      0.75      0.75        40

